# Case Study 1: Stochastic Volatility Filter (SVF)

**Circulatory Fidelity v1.1: Inference Coupling Diagnostic**

This notebook demonstrates how Inference Coupling (IC) diagnoses mean-field failure in filtering models where latent volatility modulates state dynamics.

---

## Key Concepts

- **IC = |ρ|**: The primary diagnostic for MFVI failure (Linfoot correlation)
- **Copula-based estimation**: Rank transform → probit → Pearson correlation
- **Filtering model interpretation**: High IC → MFVI will fail

---

## Model Specification

The Stochastic Volatility Filter is a three-level hierarchy:

$$
\begin{align}
x_3(t) &= x_3(t-1) + \varepsilon_3, \quad \varepsilon_3 \sim \mathcal{N}(0, \sigma_{\text{vol}}^2) \quad \text{[volatility]}\\
\sigma_2(t) &= \sigma_{\text{base}} \cdot \exp(\kappa \cdot x_3(t)) \quad \text{[coupling]}\\
x_2(t) &= x_2(t-1) + \varepsilon_2, \quad \varepsilon_2 \sim \mathcal{N}(0, \sigma_2(t)^2) \quad \text{[state]}\\
y(t) &= x_2(t) + \varepsilon_y, \quad \varepsilon_y \sim \mathcal{N}(0, \sigma_{\text{obs}}^2) \quad \text{[observation]}
\end{align}
$$

The coupling parameter $\kappa$ controls how strongly volatility modulates state dynamics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata, norm
from dataclasses import dataclass
from typing import NamedTuple, Tuple

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Core Functions: IC Estimation

**Primary diagnostic**: IC = |ρ| (Linfoot correlation)

**Unified workflow**: The copula-based estimator is recommended for **all applications**:
- **Exact for Gaussian data**: Returns |ρ| with negligible difference (<0.001) from direct Pearson
- **Conservative for non-Gaussian data**: Provides lower bound on true IC
- **No distributional verification needed**: Works correctly regardless of marginal distributions

Algorithm:
1. Rank-transform to uniform marginals
2. Apply inverse normal CDF (probit transform)
3. Compute Pearson correlation of transformed variables
4. IC = |ρ|


In [ ]:
def inference_coupling(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """
    Compute Inference Coupling (IC) using copula-based estimation.
    
    IC = |ρ| where ρ is computed from rank-transformed, probit-transformed data.
    
    Returns:
        ic: Inference Coupling in [0, 1]
        se: Standard error (Fisher transform)
    """
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    n = len(x)
    
    # Step 1: Rank transform to uniform
    u = (rankdata(x) - 0.5) / n
    v = (rankdata(y) - 0.5) / n
    
    # Step 2: Probit transform
    z_x = norm.ppf(u)
    z_y = norm.ppf(v)
    
    # Step 3: Pearson correlation
    rho = np.corrcoef(z_x, z_y)[0, 1]
    
    # IC = |ρ|
    ic = np.abs(rho)
    
    # Fisher transform SE
    se = 1.0 / np.sqrt(n - 3) if n > 3 else np.nan
    
    return ic, se

def ic_gaussian(rho: float) -> float:
    """Closed-form IC for Gaussians: IC = |ρ|"""
    return np.abs(np.clip(rho, -1.0, 1.0))

## SVF Model Implementation

In [ ]:
@dataclass
class SVFParams:
    """SVF model parameters (matching manuscript)."""
    coupling: float = 0.5           # κ: volatility-state coupling
    base_volatility: float = 0.5    # σ_base: baseline state volatility
    volatility_noise: float = 0.3   # σ_vol: volatility random walk noise
    observation_noise: float = 0.5  # σ_obs: observation noise

class SVFSimulation(NamedTuple):
    """Container for SVF simulation results."""
    x3: np.ndarray   # Volatility trajectory
    x2: np.ndarray   # State trajectory
    y: np.ndarray    # Observations
    vol: np.ndarray  # Instantaneous volatility
    params: SVFParams

def simulate_svf(params: SVFParams, T: int = 300, seed: int = None) -> SVFSimulation:
    """Simulate from SVF generative model."""
    if seed is not None:
        np.random.seed(seed)
    
    x3 = np.zeros(T)
    x2 = np.zeros(T)
    vol = np.zeros(T)
    y = np.zeros(T)
    
    vol[0] = params.base_volatility
    y[0] = np.random.normal(0, params.observation_noise)
    
    for t in range(1, T):
        x3[t] = x3[t-1] + np.random.normal(0, params.volatility_noise)
        log_vol = np.clip(params.coupling * x3[t], -3, 3)
        vol[t] = np.clip(params.base_volatility * np.exp(log_vol), 0.1, 5.0)
        x2[t] = x2[t-1] + np.random.normal(0, vol[t])
        y[t] = x2[t] + np.random.normal(0, params.observation_noise)
    
    return SVFSimulation(x3=x3, x2=x2, y=y, vol=vol, params=params)

## IC Computation for SVF

In [ ]:
def compute_ic_svf(sim: SVFSimulation) -> Tuple[float, float]:
    """
    Compute IC measuring volatility-state coupling.
    
    Uses copula-based estimation (exact for Gaussians, conservative for non-Gaussians).
    """
    x3 = sim.x3[1:]
    dx2 = np.diff(sim.x2)
    log_abs_dx2 = np.log(np.abs(dx2) + 1e-10)
    
    return inference_coupling(x3, log_abs_dx2)

## Visualization: Single Simulation

In [ ]:
params = SVFParams(coupling=1.0)
sim = simulate_svf(params, T=300, seed=42)
ic, se = compute_ic_svf(sim)

print(f"Coupling κ = {params.coupling}")
print(f"Computed IC = {ic:.4f} ± {se:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(sim.x3, 'k-', lw=0.8)
axes[0].set_title('Volatility Process $x_3(t)$')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('$x_3$')

axes[1].plot(sim.x2, 'k-', lw=0.8)
axes[1].fill_between(range(len(sim.x2)), 
                     sim.x2 - 2*sim.vol, 
                     sim.x2 + 2*sim.vol, 
                     alpha=0.3, color='gray')
axes[1].set_title('State Process $x_2(t)$ with ±2σ bands')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('$x_2$')

axes[2].scatter(sim.x3[1:], np.log(np.abs(np.diff(sim.x2)) + 1e-10), 
                alpha=0.3, s=10, c='black')
axes[2].set_title(f'Volatility-Innovation Relationship\nIC = {ic:.3f} ± {se:.3f}')
axes[2].set_xlabel('$x_3(t)$')
axes[2].set_ylabel('$\\log|Δx_2(t)|$')

plt.tight_layout()
plt.show()

## Inference Methods

In [ ]:
def mf_kalman_filter(sim: SVFSimulation):
    """Mean-field Kalman filter: ignores volatility coupling."""
    T = len(sim.y)
    avg_vol = sim.params.base_volatility
    
    x2_est = np.zeros(T)
    var_est = np.ones(T)
    
    for t in range(1, T):
        pred_var = var_est[t-1] + avg_vol**2
        obs_var = sim.params.observation_noise**2
        K = pred_var / (pred_var + obs_var)
        x2_est[t] = x2_est[t-1] + K * (sim.y[t] - x2_est[t-1])
        var_est[t] = (1 - K) * pred_var
    
    mse = np.mean((x2_est - sim.x2)**2)
    return x2_est, mse

def oracle_kalman_filter(sim: SVFSimulation):
    """Oracle Kalman filter: knows true volatility."""
    T = len(sim.y)
    
    x2_est = np.zeros(T)
    var_est = np.ones(T)
    
    for t in range(1, T):
        pred_var = var_est[t-1] + sim.vol[t]**2
        obs_var = sim.params.observation_noise**2
        K = pred_var / (pred_var + obs_var)
        x2_est[t] = x2_est[t-1] + K * (sim.y[t] - x2_est[t-1])
        var_est[t] = (1 - K) * pred_var
    
    mse = np.mean((x2_est - sim.x2)**2)
    return x2_est, mse

In [ ]:
mf_est, mf_mse = mf_kalman_filter(sim)
oracle_est, oracle_mse = oracle_kalman_filter(sim)

print(f"Mean-Field MSE:  {mf_mse:.4f}")
print(f"Oracle MSE:      {oracle_mse:.4f}")
print(f"MSE Ratio:       {mf_mse/oracle_mse:.2f}×")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sim.x2, 'k-', lw=1.5, label='True state', alpha=0.8)
ax.plot(mf_est, 'r--', lw=1, label=f'Mean-field (MSE={mf_mse:.3f})')
ax.plot(oracle_est, 'b--', lw=1, label=f'Oracle (MSE={oracle_mse:.3f})')
ax.set_xlabel('Time')
ax.set_ylabel('State')
ax.set_title(f'State Estimation Comparison (κ = {params.coupling}, IC = {ic:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## Validation Against Manuscript Data

Load the validation dataset used in the manuscript (N = 8,000 simulations).

In [ ]:
# Load manuscript validation data
try:
    df = pd.read_csv('../data/svf_validation.csv')
    print(f"Loaded validation data: {len(df)} simulations")
    print(f"\nSummary statistics:")
    print(f"  IC range: {df['ic'].min():.4f} - {df['ic'].max():.4f}")
    print(f"  MSE ratio range: {df['mse_ratio'].min():.2f} - {df['mse_ratio'].max():.2f}")
    
    # Correlation
    r = np.corrcoef(df['ic'], df['mse_ratio'])[0, 1]
    print(f"  Correlation (IC vs MSE ratio): r = {r:.3f}")
except FileNotFoundError:
    print("Validation data not found. Run parameter sweep instead.")
    df = None

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Panel A: IC vs Coupling
    grouped = df.groupby('coupling').agg({'ic': ['mean', 'std']}).reset_index()
    grouped.columns = ['coupling', 'ic_mean', 'ic_std']
    axes[0].errorbar(grouped['coupling'], grouped['ic_mean'], yerr=grouped['ic_std'],
                    fmt='o-', color='black', capsize=3)
    axes[0].axhline(0.10, color='red', linestyle='--', label='Threshold (0.10)')
    axes[0].set_xlabel('Coupling κ')
    axes[0].set_ylabel('IC')
    axes[0].set_title('(A) IC increases with coupling')
    axes[0].legend()
    
    # Panel B: MSE Ratio vs Coupling  
    grouped_mse = df.groupby('coupling').agg({'mse_ratio': ['mean', 'std']}).reset_index()
    grouped_mse.columns = ['coupling', 'mse_mean', 'mse_std']
    axes[1].errorbar(grouped_mse['coupling'], grouped_mse['mse_mean'], yerr=grouped_mse['mse_std'],
                    fmt='s-', color='black', capsize=3)
    axes[1].axhline(2.0, color='red', linestyle='--', label='Failure threshold (2×)')
    axes[1].set_xlabel('Coupling κ')
    axes[1].set_ylabel('MSE Ratio')
    axes[1].set_title('(B) MFVI degradation')
    axes[1].legend()
    
    # Panel C: IC vs MSE Ratio
    subsample = df.sample(n=min(500, len(df)), random_state=42)
    axes[2].scatter(subsample['ic'], subsample['mse_ratio'], alpha=0.3, s=15, c='gray')
    r = np.corrcoef(df['ic'], df['mse_ratio'])[0, 1]
    axes[2].set_xlabel('IC')
    axes[2].set_ylabel('MSE Ratio')
    axes[2].set_title(f'(C) IC predicts failure (r = {r:.2f})')
    axes[2].axvline(0.10, color='red', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

## Key Findings

1. **IC increases with coupling strength κ**
2. **High IC predicts MFVI failure**: Above the negligible threshold, mean-field inference degrades significantly
3. **Copula estimation provides conservative bounds** with closed-form standard errors

### Practical Recommendation (Filtering Models)

**Interpretive Scale** (from manuscript Section 2.7):

| IC Range | Coupling Regime | Interpretation |
|----------|-----------------|----------------|
| < 0.25 | Negligible | MFVI safe |
| 0.25 - 0.35 | Weak | MFVI likely acceptable |
| 0.35 - 0.55 | Moderate | Caution warranted |
| 0.55 - 0.70 | Strong | Consider structured inference |
| > 0.70 | Very strong | Structured inference required |

*Note: The threshold line at 0.10 in the plot above is a calibration reference point, not a general recommendation. Use the interpretive scale above for practical decisions.*
